# Trabalho Prático II - Inteligência Artificial

**Disciplina:** Inteligência Artificial  
**Professor:** Douglas Castilho  
**Aluna:** Maria Eduarda Batista Henrique
**Instituição:** IFSULDEMINAS - Campus Poços de Caldas  
**Data de Entrega:** 23/06/2026  

---

## Pré-processamento e Análise Preditiva

**Base de Dados:** Heart Disease (Cleveland) - UCI Machine Learning Repository  

Esta base contém dados de 303 pacientes, com 14 atributos médicos utilizados para diagnosticar a presença de doença cardíaca. O atributo alvo é `num` (diagnóstico), onde 0 indica ausência de doença e valores de 1 a 4 indicam presença de doença cardíaca em diferentes graus.

Neste trabalho, realizei o pré-processamento completo dos dados seguindo o roteiro proposto e, em seguida, construí modelos preditivos utilizando KNN, Árvore de Decisão e MLP para classificar se um paciente possui ou não doença cardíaca.


---
## Importações

Abaixo estão todas as bibliotecas que utilizei. As únicas bibliotecas externas permitidas para os classificadores são do scikit-learn. Todo o pré-processamento e cálculos estatísticos foram implementados manualmente, sem uso de pandas, numpy ou outras bibliotecas.

In [ ]:
# Importando apenas o necessário
# Para os classificadores: KNN, Decision Tree e MLP usarei scikit-learn
# Todo o resto foi implementado manualmente

import math
import random
from collections import Counter

# KNN, Árvore de Decisão e MLP - únicas bibliotecas externas permitidas
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier

---
## 1. Carregamento dos Dados

Primeiro, vou carregar o arquivo `processed.cleveland.data` manualmente. Neste dataset, valores ausentes são representados pelo caractere `?`. Optei por converter esses valores para `None` para tratá-los posteriormente.

Os 14 atributos da base são:

| Índice | Nome | Descrição |
|--------|------|-----------|
| 0 | age | Idade em anos |
| 1 | sex | Sexo (0 = feminino, 1 = masculino) |
| 2 | cp | Tipo de dor no peito (1 = angina típica, 2 = angina atípica, 3 = não anginosa, 4 = assintomático) |
| 3 | trestbps | Pressão arterial em repouso (mm Hg) |
| 4 | chol | Colesterol sérico (mg/dl) |
| 5 | fbs | Glicemia em jejum > 120 mg/dl (1 = sim, 0 = não) |
| 6 | restecg | Eletrocardiograma em repouso (0 = normal, 1 = anormalidade ST-T, 2 = hipertrofia ventricular esquerda) |
| 7 | thalach | Frequência cardíaca máxima alcançada |
| 8 | exang | Angina induzida por exercício (1 = sim, 0 = não) |
| 9 | oldpeak | Depressão do segmento ST induzida por exercício |
| 10 | slope | Inclinação do segmento ST (1 = ascendente, 2 = plano, 3 = descendente) |
| 11 | ca | Número de vasos principais coloridos por fluoroscopia (0-3) |
| 12 | thal | Talassemia (3 = normal, 6 = defeito fixo, 7 = defeito reversível) |
| 13 | num | Diagnóstico de doença cardíaca (0 = ausente, 1-4 = presente) |

In [ ]:
# Carregando os dados manualmente, linha por linha
# O arquivo CSV não tem cabeçalho, então vou adicionar os nomes das colunas manualmente

NOME_COLUNAS = [
    'age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg',
    'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'num'
]

def carregar_dados(caminho):
    """Carrega o arquivo CSV manualmente e retorna uma lista de instâncias."""
    dados = []
    with open(caminho, 'r') as arquivo:
        for linha in arquivo:
            linha = linha.strip()
            if not linha:
                continue
            valores = linha.split(',')
            instancia = []
            for v in valores:
                v = v.strip()
                if v == '?':
                    # Valor faltante: armazeno como None para tratar depois
                    instancia.append(None)
                else:
                    # Todos os atributos são numéricos, então converto para float
                    instancia.append(float(v))
            dados.append(instancia)
    return dados

# Carregando os dados
import os
# Procura o arquivo no Kaggle (/kaggle/input) ou na pasta local
caminho_dados = os.path.join('data', 'processed.cleveland.data')
for raiz, _, arquivos in os.walk('/kaggle/input'):
    if 'processed.cleveland.data' in arquivos:
        caminho_dados = os.path.join(raiz, 'processed.cleveland.data')
        break
dados = carregar_dados(caminho_dados)

print(f'Total de instâncias carregadas: {len(dados)}')
print(f'Total de atributos por instância: {len(dados[0])}')
print()
print('Primeiras 5 instâncias:')
for i in range(5):
    print(dados[i])

Total de instâncias carregadas: 303
Total de atributos por instância: 14

Primeiras 5 instâncias:
[63.0, 1.0, 1.0, 145.0, 233.0, 1.0, 2.0, 150.0, 0.0, 2.3, 3.0, 0.0, 6.0, 0.0]
[67.0, 1.0, 4.0, 160.0, 286.0, 0.0, 2.0, 108.0, 1.0, 1.5, 2.0, 3.0, 3.0, 2.0]
[67.0, 1.0, 4.0, 120.0, 229.0, 0.0, 2.0, 129.0, 1.0, 2.6, 2.0, 2.0, 7.0, 1.0]
[37.0, 1.0, 3.0, 130.0, 250.0, 0.0, 0.0, 187.0, 0.0, 3.5, 3.0, 0.0, 3.0, 0.0]
[41.0, 0.0, 2.0, 130.0, 204.0, 0.0, 2.0, 172.0, 0.0, 1.4, 1.0, 0.0, 3.0, 0.0]


---
## 2. Identificação do Atributo Alvo

O atributo alvo é a coluna `num` (índice 13), que representa o diagnóstico de doença cardíaca. 

**Valores originais:**
- 0: ausência de doença cardíaca (estenose < 50%)
- 1, 2, 3, 4: presença de doença cardíaca (estenose > 50%), em diferentes graus

Seguindo a prática adotada pela literatura (Detrano et al., 1989; Aha & Kibler), vou converter o alvo para binário:
- **0** -> saudável (sem doença)
- **1, 2, 3, 4** -> doente (com doença)

Isso porque o objetivo principal é distinguir a presença ou ausência da doença, não o grau dela. A binarização será feita na etapa de pré-processamento, antes da modelagem.

In [ ]:
# Contando a distribuição original do alvo
contagem_original = Counter()
for inst in dados:
    contagem_original[inst[13]] += 1

print('Distribuição original do atributo alvo (num):')
for valor in sorted(contagem_original.keys()):
    print(f'  {int(valor)}: {contagem_original[valor]} pacientes')
print()
print(f'Total: {sum(contagem_original.values())} pacientes')
print()
print('A maioria dos experimentos na literatura (Detrano, Aha & Kibler, Gennari)')
print('utilizam a classificação binária: 0 (saudável) vs 1-4 (doente).')
print('Portanto, seguirei essa abordagem na etapa de modelagem.')

Distribuição original do atributo alvo (num):
  0: 164 pacientes
  1: 55 pacientes
  2: 36 pacientes
  3: 35 pacientes
  4: 13 pacientes

Total: 303 pacientes

A maioria dos experimentos na literatura (Detrano, Aha & Kibler, Gennari)
utilizam a classificação binária: 0 (saudável) vs 1-4 (doente).
Portanto, seguirei essa abordagem na etapa de modelagem.


---
## 3. Identificação dos Tipos e Escalas dos Atributos

Para cada atributo de entrada, classifiquei:
- **Tipo:** Quantitativo (numérico) ou Qualitativo (categórico)
- **Escala de medida:** Nominal, Ordinal, Intervalar ou Racional

| Atributo | Tipo | Escala | Justificativa |
|----------|------|--------|---------------|
| age | Quantitativo | Racional | Idade em anos, zero absoluto, diferenças proporcionais |
| sex | Qualitativo | Nominal | Categorias sem ordem (masculino/feminino) |
| cp | Qualitativo | Nominal | Tipos de dor sem ordem hierárquica |
| trestbps | Quantitativo | Racional | Pressão em mm Hg, zero absoluto |
| chol | Quantitativo | Racional | Colesterol em mg/dl, zero absoluto |
| fbs | Qualitativo | Nominal | Binário (sim/não), sem ordem |
| restecg | Qualitativo | Nominal | Categorias de ECG sem ordem linear |
| thalach | Quantitativo | Racional | Frequência cardíaca, zero absoluto |
| exang | Qualitativo | Nominal | Binário (sim/não), sem ordem |
| oldpeak | Quantitativo | Racional | Depressão ST em mm, zero absoluto |
| slope | Qualitativo | Ordinal | Ascendente (1) < Plano (2) < Descendente (3) |
| ca | Quantitativo | Racional | Contagem de vasos, zero absoluto (discreto) |
| thal | Qualitativo | Nominal | Categorias sem ordem linear |

**Observação sobre o slope:** Diferente dos demais atributos qualitativos, o `slope` possui uma ordem natural: ascendente -> plano -> descendente. Por isso classifiquei como ordinal.

In [ ]:
# Separando os índices dos atributos por tipo para uso posterior
# Essa classificação foi feita com base na documentação da base de dados

# Atributos quantitativos (numéricos contínuos ou discretos)
# age, trestbps, chol, thalach, oldpeak, ca
INDICES_QUANTITATIVOS = [0, 3, 4, 7, 9, 11]

# Atributos qualitativos (categóricos, incluindo binários)
# sex, cp, fbs, restecg, exang, slope, thal
INDICES_QUALITATIVOS = [1, 2, 5, 6, 8, 10, 12]

# Atributos ordinais (têm ordem, mas não são numéricos contínuos)
# slope: 1=upsloping < 2=flat < 3=downsloping
INDICES_ORIDINAIS = [10]

# Atributos nominais (sem ordem)
# sex, cp, fbs, restecg, exang, thal
INDICES_NOMINAIS = [1, 2, 5, 6, 8, 12]

# Atributo alvo
INDICE_ALVO = 13

print('Classificação dos atributos concluída.')
print(f'Quantitativos: {[NOME_COLUNAS[i] for i in INDICES_QUANTITATIVOS]}')
print(f'Qualitativos nominais: {[NOME_COLUNAS[i] for i in INDICES_NOMINAIS]}')
print(f'Qualitativos ordinais: {[NOME_COLUNAS[i] for i in INDICES_ORIDINAIS]}')

Classificação dos atributos concluída.
Quantitativos: ['age', 'trestbps', 'chol', 'thalach', 'oldpeak', 'ca']
Qualitativos nominais: ['sex', 'cp', 'fbs', 'restecg', 'exang', 'thal']
Qualitativos ordinais: ['slope']


---
## 4. Funções Estatísticas (Implementação Manual)

Antes de calcular as medidas exploratórias, implementei manualmente as funções estatísticas básicas. Não utilizei numpy nem pandas — tudo foi feito com Python puro.

As funções implementadas:
- **media**: soma dos valores dividida pela quantidade
- **mediana**: valor central dos dados ordenados
- **moda**: valor que mais se repete
- **variancia**: média dos quadrados das diferenças em relação à média (usei n-1 para amostra)
- **desvio_padrao**: raiz quadrada da variância
- **quartis**: Q1 (25º percentil), Q2 (mediana), Q3 (75º percentil)
- **amplitude**: valor máximo - valor mínimo

In [ ]:
def media(valores):
    """Calcula a média aritmética.
    Fórmula: soma dos valores / quantidade de valores"""
    soma = 0
    for v in valores:
        soma += v
    return soma / len(valores)

def mediana(valores):
    """Calcula a mediana.
    Ordeno os valores e pego o do meio.
    Se a quantidade for par, faço a média dos dois centrais."""
    ordenados = sorted(valores)
    n = len(ordenados)
    meio = n // 2
    if n % 2 == 0:
        # Par: média dos dois valores centrais
        return (ordenados[meio - 1] + ordenados[meio]) / 2
    else:
        # Ímpar: valor do meio
        return ordenados[meio]

def moda(valores):
    """Calcula a moda (valor mais frequente).
    Uso Counter para contar as frequências."""
    contagem = Counter(valores)
    # most_common(1) retorna [(valor, frequencia)]
    return contagem.most_common(1)[0][0]

def variancia(valores):
    """Calcula a variância amostral.
    Fórmula: soma((xi - media)^2) / (n - 1)
    Uso n-1 (correção de Bessel) porque é uma amostra, não a população toda."""
    med = media(valores)
    soma_quadrados = 0
    for v in valores:
        soma_quadrados += (v - med) ** 2
    return soma_quadrados / (len(valores) - 1)

def desvio_padrao(valores):
    """Calcula o desvio padrão amostral.
    É a raiz quadrada da variância. Volta para a unidade original dos dados."""
    return math.sqrt(variancia(valores))

def amplitude(valores):
    """Calcula a amplitude total (max - min)."""
    return max(valores) - min(valores)

def quartis(valores):
    """Calcula Q1, Q2 (mediana) e Q3.
    Uso o método de interpolação linear padrão:
    - Q1: mediana da metade inferior dos dados
    - Q2: mediana
    - Q3: mediana da metade superior dos dados"""
    ordenados = sorted(valores)
    n = len(ordenados)
    q2 = mediana(ordenados)
    
    # Metade inferior (para Q1)
    metade_inferior = ordenados[:n // 2]
    q1 = mediana(metade_inferior)
    
    # Metade superior (para Q3)
    if n % 2 == 0:
        metade_superior = ordenados[n // 2:]
    else:
        metade_superior = ordenados[n // 2 + 1:]
    q3 = mediana(metade_superior)
    
    return q1, q2, q3

def minimo(valores):
    return min(valores)

def maximo(valores):
    return max(valores)

def iqr(valores):
    """Calcula o Intervalo Interquartil (Q3 - Q1)."""
    q1, _, q3 = quartis(valores)
    return q3 - q1

# Testando as funções com um exemplo simples
teste = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
print('Teste das funções estatísticas:')
print(f'  Valores: {teste}')
print(f'  Média: {media(teste):.2f}')
print(f'  Mediana: {mediana(teste)}')
print(f'  Variância: {variancia(teste):.2f}')
print(f'  Desvio Padrão: {desvio_padrao(teste):.2f}')
print(f'  Amplitude: {amplitude(teste)}')
print(f'  Q1, Q2, Q3: {quartis(teste)}')
print(f'  IQR: {iqr(teste):.2f}')

Teste das funções estatísticas:
  Valores: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
  Média: 5.50
  Mediana: 5.5
  Variância: 9.17
  Desvio Padrão: 3.03
  Amplitude: 9
  Q1, Q2, Q3: (3, 5.5, 8)
  IQR: 5.00


---
## 5. Medidas de Localidade (Tendência Central)

Nesta seção, calculo as medidas de localidade para cada atributo quantitativo:
- **Média**: ponto de equilíbrio dos dados
- **Mediana**: valor central, menos sensível a outliers que a média
- **Moda**: valor mais frequente

Para cada atributo, explico o que o resultado significa no contexto médico.

In [ ]:
# Extraio os valores de cada atributo quantitativo, ignorando None
def extrair_coluna(dados, indice):
    """Extrai uma coluna dos dados, pulando valores None."""
    valores = []
    for inst in dados:
        if inst[indice] is not None:
            valores.append(inst[indice])
    return valores

print('=' * 70)
print('MEDIDAS DE LOCALIDADE (TENDÊNCIA CENTRAL)')
print('=' * 70)
print()

for i in INDICES_QUANTITATIVOS:
    valores = extrair_coluna(dados, i)
    nome = NOME_COLUNAS[i]
    
    med = media(valores)
    medn = mediana(valores)
    mod = moda(valores)
    
    print(f'{nome} ({len(valores)} valores válidos):')
    print(f'  Média...: {med:.2f}')
    print(f'  Mediana.: {medn:.2f}')
    print(f'  Moda....: {mod:.2f}')
    
    # Interpretação dos resultados
    if nome == 'age':
        print(f'  -> A idade média dos pacientes é de aproximadamente {med:.0f} anos.')
        if abs(med - medn) < 1:
            print(f'  -> Média e mediana próximas, indicando distribuição simétrica.')
    elif nome == 'trestbps':
        print(f'  -> A pressão arterial média em repouso é de {med:.0f} mm Hg.')
    elif nome == 'chol':
        print(f'  -> O colesterol médio é de {med:.0f} mg/dl.')
    elif nome == 'thalach':
        print(f'  -> A frequência cardíaca máxima média é de {med:.0f} bpm.')
    elif nome == 'oldpeak':
        print(f'  -> A depressão ST média é de {med:.2f} mm.')
    elif nome == 'ca':
        print(f'  -> A média de vasos coloridos é de {med:.2f}.')
    print()

MEDIDAS DE LOCALIDADE (TENDÊNCIA CENTRAL)

age (303 valores válidos):
  Média...: 54.44
  Mediana.: 56.00
  Moda....: 58.00
  -> A idade média dos pacientes é de aproximadamente 54 anos.

trestbps (303 valores válidos):
  Média...: 131.69
  Mediana.: 130.00
  Moda....: 120.00
  -> A pressão arterial média em repouso é de 132 mm Hg.

chol (303 valores válidos):
  Média...: 246.69
  Mediana.: 241.00
  Moda....: 204.00
  -> O colesterol médio é de 247 mg/dl.

thalach (303 valores válidos):
  Média...: 149.61
  Mediana.: 153.00
  Moda....: 162.00
  -> A frequência cardíaca máxima média é de 150 bpm.

oldpeak (303 valores válidos):
  Média...: 1.04
  Mediana.: 0.80
  Moda....: 0.00
  -> A depressão ST média é de 1.04 mm.

ca (299 valores válidos):
  Média...: 0.67
  Mediana.: 0.00
  Moda....: 0.00
  -> A média de vasos coloridos é de 0.67.



---
## 6. Medidas de Espalhamento (Dispersão)

Agora calculo as medidas que indicam o quanto os dados estão espalhados em torno da média:
- **Amplitude**: diferença entre o maior e o menor valor
- **Variância**: média dos quadrados dos desvios em relação à média
- **Desvio Padrão**: raiz quadrada da variância (interpretável na unidade original)

Uma variância/desvio padrão alto indica que os pacientes são muito diferentes entre si naquele atributo; um valor baixo indica que são mais homogêneos.

In [ ]:
print('=' * 70)
print('MEDIDAS DE ESPALHAMENTO (DISPERSÃO)')
print('=' * 70)
print()

for i in INDICES_QUANTITATIVOS:
    valores = extrair_coluna(dados, i)
    nome = NOME_COLUNAS[i]
    
    amp = amplitude(valores)
    var = variancia(valores)
    dp = desvio_padrao(valores)
    
    print(f'{nome}:')
    print(f'  Amplitude......: {amp:.2f}')
    print(f'  Variância......: {var:.2f}')
    print(f'  Desvio Padrão..: {dp:.2f}')
    
    # Interpretação
    med = media(valores)
    cv = (dp / med) * 100  # Coeficiente de variação
    print(f'  Coef. Variação.: {cv:.1f}%')
    if cv < 30:
        print(f'  -> Baixa dispersão relativa (dados homogêneos)')
    elif cv < 60:
        print(f'  -> Média dispersão relativa')
    else:
        print(f'  -> Alta dispersão relativa (dados heterogêneos)')
    print()

MEDIDAS DE ESPALHAMENTO (DISPERSÃO)

age:
  Amplitude......: 48.00
  Variância......: 81.70
  Desvio Padrão..: 9.04
  Coef. Variação.: 16.6%
  -> Baixa dispersão relativa (dados homogêneos)

trestbps:
  Amplitude......: 106.00
  Variância......: 309.75
  Desvio Padrão..: 17.60
  Coef. Variação.: 13.4%
  -> Baixa dispersão relativa (dados homogêneos)

chol:
  Amplitude......: 438.00
  Variância......: 2680.85
  Desvio Padrão..: 51.78
  Coef. Variação.: 21.0%
  -> Baixa dispersão relativa (dados homogêneos)

thalach:
  Amplitude......: 131.00
  Variância......: 523.27
  Desvio Padrão..: 22.88
  Coef. Variação.: 15.3%
  -> Baixa dispersão relativa (dados homogêneos)

oldpeak:
  Amplitude......: 6.20
  Variância......: 1.35
  Desvio Padrão..: 1.16
  Coef. Variação.: 111.7%
  -> Alta dispersão relativa (dados heterogêneos)

ca:
  Amplitude......: 3.00
  Variância......: 0.88
  Desvio Padrão..: 0.94
  Coef. Variação.: 139.4%
  -> Alta dispersão relativa (dados heterogêneos)



---
## 7. Medidas de Distribuição

Aqui calculo os quartis e o IQR para entender como os dados se distribuem ao longo do intervalo. 
- **Q1 (25%)**: 25% dos pacientes estão abaixo deste valor
- **Q2 (50%)**: mediana
- **Q3 (75%)**: 75% dos pacientes estão abaixo deste valor
- **IQR (Q3 - Q1)**: contém 50% dos dados centrais

In [ ]:
print('=' * 70)
print('MEDIDAS DE DISTRIBUIÇÃO (QUARTIS E IQR)')
print('=' * 70)
print()

for i in INDICES_QUANTITATIVOS:
    valores = extrair_coluna(dados, i)
    nome = NOME_COLUNAS[i]
    
    q1, q2, q3 = quartis(valores)
    minimo_val = minimo(valores)
    maximo_val = maximo(valores)
    iqr_val = q3 - q1
    
    print(f'{nome}:')
    print(f'  Mínimo.: {minimo_val:.2f}')
    print(f'  Q1.....: {q1:.2f}')
    print(f'  Q2..... : {q2:.2f}  (mediana)')
    print(f'  Q3.....: {q3:.2f}')
    print(f'  Máximo.: {maximo_val:.2f}')
    print(f'  IQR....: {iqr_val:.2f}')
    print()

MEDIDAS DE DISTRIBUIÇÃO (QUARTIS E IQR)

age:
  Mínimo.: 29.00
  Q1.....: 48.00
  Q2..... : 56.00  (mediana)
  Q3.....: 61.00
  Máximo.: 77.00
  IQR....: 13.00

trestbps:
  Mínimo.: 94.00
  Q1.....: 120.00
  Q2..... : 130.00  (mediana)
  Q3.....: 140.00
  Máximo.: 200.00
  IQR....: 20.00

chol:
  Mínimo.: 126.00
  Q1.....: 211.00
  Q2..... : 241.00  (mediana)
  Q3.....: 275.00
  Máximo.: 564.00
  IQR....: 64.00

thalach:
  Mínimo.: 71.00
  Q1.....: 133.00
  Q2..... : 153.00  (mediana)
  Q3.....: 166.00
  Máximo.: 202.00
  IQR....: 33.00

oldpeak:
  Mínimo.: 0.00
  Q1.....: 0.00
  Q2..... : 0.80  (mediana)
  Q3.....: 1.60
  Máximo.: 6.20
  IQR....: 1.60

ca:
  Mínimo.: 0.00
  Q1.....: 0.00
  Q2..... : 0.00  (mediana)
  Q3.....: 1.00
  Máximo.: 3.00
  IQR....: 1.00



---
## 8. Atributos Qualitativos — Frequências

Para atributos qualitativos (categóricos), medidas como média e mediana não fazem sentido. Em vez disso, apresento a **tabela de frequências** de cada categoria, que mostra quantos pacientes pertencem a cada classe do atributo.

In [ ]:
# Dicionário com a descrição dos valores para alguns atributos qualitativos
DESCRICOES_VALORES = {
    'sex': {0.0: 'Feminino', 1.0: 'Masculino'},
    'cp': {1.0: 'Angina típica', 2.0: 'Angina atípica', 3.0: 'Não anginosa', 4.0: 'Assintomático'},
    'fbs': {0.0: 'Glicemia <= 120', 1.0: 'Glicemia > 120'},
    'restecg': {0.0: 'Normal', 1.0: 'Anormalidade ST-T', 2.0: 'Hipertrofia VE'},
    'exang': {0.0: 'Não', 1.0: 'Sim'},
    'slope': {1.0: 'Ascendente', 2.0: 'Plano', 3.0: 'Descendente'},
    'thal': {3.0: 'Normal', 6.0: 'Defeito fixo', 7.0: 'Defeito reversível'}
}

print('=' * 70)
print('TABELAS DE FREQUÊNCIA - ATRIBUTOS QUALITATIVOS')
print('=' * 70)
print()

for i in INDICES_QUALITATIVOS:
    valores = extrair_coluna(dados, i)
    nome = NOME_COLUNAS[i]
    contagem = Counter(valores)
    total = len(valores)
    
    print(f'{nome}:')
    for valor in sorted(contagem.keys()):
        freq = contagem[valor]
        pct = (freq / total) * 100
        descricao = DESCRICOES_VALORES.get(nome, {}).get(valor, f'Valor {int(valor)}')
        barra = '*' * int(pct / 2)
        print(f'  {descricao:25s}: {freq:3d} ({pct:5.1f}%) {barra}')
    print()

TABELAS DE FREQUÊNCIA - ATRIBUTOS QUALITATIVOS

sex:
  Feminino                 :  97 ( 32.0%) ****************
  Masculino                : 206 ( 68.0%) *********************************

cp:
  Angina típica            :  23 (  7.6%) ***
  Angina atípica           :  50 ( 16.5%) ********
  Não anginosa             :  86 ( 28.4%) **************
  Assintomático            : 144 ( 47.5%) ***********************

fbs:
  Glicemia <= 120          : 258 ( 85.1%) ******************************************
  Glicemia > 120           :  45 ( 14.9%) *******

restecg:
  Normal                   : 151 ( 49.8%) ************************
  Anormalidade ST-T        :   4 (  1.3%) 
  Hipertrofia VE           : 148 ( 48.8%) ************************

exang:
  Não                      : 204 ( 67.3%) *********************************
  Sim                      :  99 ( 32.7%) ****************

slope:
  Ascendente               : 142 ( 46.9%) ***********************
  Plano                    : 140 ( 46.2%) 

---
## 9. Identificação e Tratamento de Valores Ausentes

A base de dados Cleveland contém valores faltantes representados por `?`. Vou identificar em quais atributos eles ocorrem e tratá-los adequadamente.

**Estratégia adotada:**
- Para `ca` (contagem de vasos, 0-3): usei a **mediana**, pois é uma medida robusta para dados discretos com possíveis outliers
- Para `thal` (categórico nominal: 3, 6, 7): usei a **moda**, que é a medida mais apropriada para dados categóricos

Optei por não remover as linhas com valores faltantes porque perderíamos amostras importantes (são apenas 6 no total). O preenchimento permite manter o tamanho da base para a modelagem.

In [ ]:
def contar_faltantes(dados):
    """Conta quantos valores None existem em cada coluna."""
    faltantes = [0] * len(dados[0])
    for inst in dados:
        for j in range(len(inst)):
            if inst[j] is None:
                faltantes[j] += 1
    return faltantes

faltantes = contar_faltantes(dados)

print('Valores ausentes por atributo:')
print()
for j in range(len(NOME_COLUNAS)):
    if faltantes[j] > 0:
        print(f'  {NOME_COLUNAS[j]:12s} (col {j}): {faltantes[j]} valores ausentes')

print()
print(f'Total de valores ausentes: {sum(faltantes)}')
print(f'Total de instâncias com algum valor ausente: ', end='')

inst_com_faltantes = sum(1 for inst in dados if any(v is None for v in inst))
print(inst_com_faltantes)
print()
print('Apenas os atributos ca e thal possuem valores ausentes.')
print('Vou tratá-los a seguir.')

Valores ausentes por atributo:

  ca           (col 11): 4 valores ausentes
  thal         (col 12): 2 valores ausentes

Total de valores ausentes: 6
Total de instâncias com algum valor ausente: 6

Apenas os atributos ca e thal possuem valores ausentes.
Vou tratá-los a seguir.


In [ ]:
def preencher_faltantes(dados):
    """Preenche valores ausentes.
    - ca (col 11): mediana (é contagem discreta, mediana é robusta)
    - thal (col 12): moda (é categórico nominal, moda é apropriada)"""
    
    # Conto quantos faltantes tem em cada coluna
    faltantes = [0] * len(dados[0])
    for inst in dados:
        for j in range(len(inst)):
            if inst[j] is None:
                faltantes[j] += 1

    # Estratégia para cada coluna com faltantes
    estrategias = {
        11: 'mediana',  # ca: contagem 0-3, mediana é mais robusta
        12: 'moda'      # thal: categórico (3,6,7), moda é apropriada
    }
    
    for col, estrategia in estrategias.items():
        # Coletando valores não nulos da coluna
        valores_validos = [inst[col] for inst in dados if inst[col] is not None]
        
        if estrategia == 'mediana':
            valor_preenchimento = mediana(valores_validos)
            print(f'{NOME_COLUNAS[col]}: preenchendo {faltantes[col]} valores ausentes com a MEDIANA = {valor_preenchimento:.2f}')
            print(f'  -> Usei mediana porque ca é uma contagem discreta (0-3) e a mediana é')
            print(f'    menos sensível a outliers do que a média.')
        elif estrategia == 'moda':
            valor_preenchimento = moda(valores_validos)
            desc = DESCRICOES_VALORES.get('thal', {}).get(valor_preenchimento, str(int(valor_preenchimento)))
            print(f'{NOME_COLUNAS[col]}: preenchendo {faltantes[col]} valores ausentes com a MODA = {int(valor_preenchimento)} ({desc})')
            print(f'  -> Usei moda porque thal é um atributo categórico nominal,')
            print(f'    e a moda é a medida de tendência central adequada para esse tipo.')
        
        # Preenchendo
        for inst in dados:
            if inst[col] is None:
                inst[col] = valor_preenchimento
    
    return dados

print('Preenchendo valores ausentes...')
print()
dados = preencher_faltantes(dados)
print()
print('Verificando se ainda há valores ausentes...')
faltantes_apos = contar_faltantes(dados)
if sum(faltantes_apos) == 0:
    print('  [OK] Todos os valores ausentes foram preenchidos com sucesso!')
else:
    print(f'   Ainda há {sum(faltantes_apos)} valores ausentes.')

Preenchendo valores ausentes...

ca: preenchendo 4 valores ausentes com a MEDIANA = 0.00
  -> Usei mediana porque ca é uma contagem discreta (0-3) e a mediana é
    menos sensível a outliers do que a média.
thal: preenchendo 2 valores ausentes com a MODA = 3 (Normal)
  -> Usei moda porque thal é um atributo categórico nominal,
    e a moda é a medida de tendência central adequada para esse tipo.

Verificando se ainda há valores ausentes...
  [OK] Todos os valores ausentes foram preenchidos com sucesso!


---
## 10. Identificação de Outliers

Para detectar outliers nos atributos quantitativos, utilizei o **método do IQR (Intervalo Interquartil)**, também conhecido como **regra de Tukey**:

- **Limite inferior** = Q1 - 1.5 × IQR
- **Limite superior** = Q3 + 1.5 × IQR

Valores fora desses limites são considerados outliers. O fator 1.5 é o padrão recomendado por Tukey (1977).

**Decisão:** Apenas identifiquei e reportei os outliers, mas **não os removi**. Como se trata de dados médicos reais, cada paciente é um caso válido — um paciente com pressão muito alta ou colesterol muito baixo não é um erro, é uma condição real que pode ser relevante para o diagnóstico.

In [ ]:
def detectar_outliers_iqr(valores):
    """Detecta outliers usando o método IQR.
    Retorna os índices dos valores que estão fora dos limites."""
    q1, _, q3 = quartis(valores)
    iqr_val = q3 - q1
    lim_inf = q1 - 1.5 * iqr_val
    lim_sup = q3 + 1.5 * iqr_val
    
    outliers = []
    for idx, v in enumerate(valores):
        if v < lim_inf or v > lim_sup:
            outliers.append((idx, v))
    
    return outliers, lim_inf, lim_sup

print('=' * 70)
print('DETECÇÃO DE OUTLIERS - MÉTODO IQR')
print('=' * 70)
print()

for i in INDICES_QUANTITATIVOS:
    valores = extrair_coluna(dados, i)
    nome = NOME_COLUNAS[i]
    
    outliers, lim_inf, lim_sup = detectar_outliers_iqr(valores)
    
    print(f'{nome}:')
    print(f'  Limites: [{lim_inf:.2f}, {lim_sup:.2f}]')
    print(f'  Outliers encontrados: {len(outliers)} de {len(valores)} ({len(outliers)/len(valores)*100:.1f}%)')
    
    if outliers:
        # Mostro os 5 outliers mais extremos para não poluir a saída
        outliers_ordenados = sorted(outliers, key=lambda x: abs(x[1] - (lim_inf + lim_sup)/2), reverse=True)
        for idx, val in outliers_ordenados[:5]:
            tipo = 'abaixo' if val < lim_inf else 'acima'
            print(f'    Instância {idx}: valor {val:.2f} ({tipo} do limite)')
        if len(outliers) > 5:
            print(f'    ... e mais {len(outliers) - 5} outlier(s)')
    
    print('  -> Decisão: manter outliers (dados médicos reais, cada paciente é um caso válido)')
    print()

DETECÇÃO DE OUTLIERS - MÉTODO IQR

age:
  Limites: [28.50, 80.50]
  Outliers encontrados: 0 de 303 (0.0%)
  -> Decisão: manter outliers (dados médicos reais, cada paciente é um caso válido)

trestbps:
  Limites: [90.00, 170.00]
  Outliers encontrados: 9 de 303 (3.0%)
    Instância 126: valor 200.00 (acima do limite)
    Instância 188: valor 192.00 (acima do limite)
    Instância 83: valor 180.00 (acima do limite)
    Instância 201: valor 180.00 (acima do limite)
    Instância 231: valor 180.00 (acima do limite)
    ... e mais 4 outlier(s)
  -> Decisão: manter outliers (dados médicos reais, cada paciente é um caso válido)

chol:
  Limites: [115.00, 371.00]
  Outliers encontrados: 5 de 303 (1.7%)
    Instância 152: valor 564.00 (acima do limite)
    Instância 48: valor 417.00 (acima do limite)
    Instância 181: valor 409.00 (acima do limite)
    Instância 121: valor 407.00 (acima do limite)
    Instância 173: valor 394.00 (acima do limite)
  -> Decisão: manter outliers (dados médicos re

---
## 11. Dados Redundantes e Inconsistentes

Vou verificar se existem instâncias duplicadas (exatamente iguais) na base de dados. Dados redundantes podem enviesar o modelo, dando peso extra a certos padrões.

In [ ]:
def encontrar_duplicatas(dados):
    """Encontra instâncias duplicadas na base."""
    vistos = {}
    duplicatas = []
    for idx, inst in enumerate(dados):
        # Converte para tupla para poder usar como chave de dicionário
        chave = tuple(inst)
        if chave in vistos:
            duplicatas.append((idx, vistos[chave]))
        else:
            vistos[chave] = idx
    return duplicatas

duplicatas = encontrar_duplicatas(dados)

if duplicatas:
    print(f'Foram encontradas {len(duplicatas)} instâncias duplicadas:')
    for dup, orig in duplicatas[:5]:
        print(f'  Instância {dup} é duplicata da instância {orig}')
    if len(duplicatas) > 5:
        print(f'  ... e mais {len(duplicatas) - 5}')
    print()
    print('Optei por remover as duplicatas para não enviesar o modelo.')
else:
    print('[OK] Nenhuma instância duplicada encontrada na base Cleveland.')
    print('Isso já era esperado, pois a base foi organizada pelo UCI sem duplicatas.')

[OK] Nenhuma instância duplicada encontrada na base Cleveland.
Isso já era esperado, pois a base foi organizada pelo UCI sem duplicatas.


---
## 12. Análise de Desbalanceamento

Vou verificar se as classes (saudável vs doente) estão balanceadas após a binarização do alvo. Um desbalanceamento muito grande pode fazer o modelo aprender apenas a classe majoritária, por isso é importante identificar e, se necessário, aplicar técnicas de balanceamento.

In [ ]:
# Contando a distribuição binária
# 0 = saudável, 1-4 = doente
saudaveis = 0
doentes = 0
for inst in dados:
    if inst[INDICE_ALVO] == 0:
        saudaveis += 1
    else:
        doentes += 1

total = saudaveis + doentes

print('Distribuição das classes (alvo binário):')
print()
print(f'  0 - Saudável: {saudaveis} ({saudaveis/total*100:.1f}%)')
print(f'  1 - Doente..: {doentes} ({doentes/total*100:.1f}%)')
print()
print(f'  Total: {total}')
print()

# Calculando a proporção
razao = max(saudaveis, doentes) / min(saudaveis, doentes)
print(f'  Proporção (maior/menor): {razao:.2f}x')
print()

if razao < 1.5:
    print('As classes são relativamente balanceadas.')
    print('Não será necessário aplicar técnicas de balanceamento (SMOTE, subamostragem, etc).')
elif razao < 3:
    print('Há um desbalanceamento moderado.')
    print('Vou avaliar se o baseline (classe majoritária) tem desempenho muito diferente.')
else:
    print(' Desbalanceamento significativo detectado.')
    print('Seria necessário aplicar técnicas como SMOTE ou subamostragem.')

Distribuição das classes (alvo binário):

  0 - Saudável: 164 (54.1%)
  1 - Doente..: 139 (45.9%)

  Total: 303

  Proporção (maior/menor): 1.18x

As classes são relativamente balanceadas.
Não será necessário aplicar técnicas de balanceamento (SMOTE, subamostragem, etc).


---
## 13. Normalização dos Dados

Apliquei a **normalização Min-Max** nos atributos quantitativos, que transforma os valores para uma escala entre 0 e 1:

$$x_{norm} = \frac{x - x_{min}}{x_{max} - x_{min}}$$

**Por que normalizar?**
- O KNN calcula distâncias entre instâncias. Atributos com escalas maiores (ex: colesterol com valores ~200-400) dominariam o cálculo da distância sobre atributos com escalas menores (ex: ca com valores 0-3).
- A normalização coloca todos os atributos na mesma escala, permitindo que cada um contribua igualmente.
- Para Árvore de Decisão e MLP, a normalização não é estritamente necessária, mas também não prejudica.

**Por que Min-Max e não Z-Score?**
- Min-Max preserva a distribuição original dos dados
- Mantém valores em um intervalo fixo [0,1], facilitando a interpretação
- Z-score (padronização) seria mais adequado se houvesse muitos outliers, mas como mantive os outliers, Min-Max ainda é uma boa escolha

In [ ]:
def normalizar_minmax(dados, indices):
    """Normaliza os atributos nos índices especificados para [0, 1].
    Fórmula: (x - min) / (max - min)"""
    
    dados_norm = [inst[:] for inst in dados]  # Cópia para não modificar original
    
    for i in indices:
        valores = [inst[i] for inst in dados_norm]
        min_val = minimo(valores)
        max_val = maximo(valores)
        
        for inst in dados_norm:
            if max_val - min_val > 0:
                inst[i] = (inst[i] - min_val) / (max_val - min_val)
            else:
                inst[i] = 0.0  # Caso todos os valores sejam iguais
    
    return dados_norm

print('Antes da normalização (primeira instância):')
for i in INDICES_QUANTITATIVOS:
    print(f'  {NOME_COLUNAS[i]:12s} = {dados[0][i]:.2f}')

dados_norm = normalizar_minmax(dados, INDICES_QUANTITATIVOS)

print()
print('Depois da normalização (primeira instância):')
for i in INDICES_QUANTITATIVOS:
    print(f'  {NOME_COLUNAS[i]:12s} = {dados_norm[0][i]:.4f}')

print()
print('Atributos quantitativos normalizados para [0, 1].')
print('Atributos qualitativos (sex, cp, fbs, restecg, exang, slope, thal) não foram normalizados')

Antes da normalização (primeira instância):
  age          = 63.00
  trestbps     = 145.00
  chol         = 233.00
  thalach      = 150.00
  oldpeak      = 2.30
  ca           = 0.00

Depois da normalização (primeira instância):
  age          = 0.7083
  trestbps     = 0.4811
  chol         = 0.2443
  thalach      = 0.6031
  oldpeak      = 0.3710
  ca           = 0.0000

Atributos quantitativos normalizados para [0, 1].
Atributos qualitativos (sex, cp, fbs, restecg, exang, slope, thal) não foram normalizados


---
## 14. Redução de Dimensionalidade

A base Cleveland já utiliza apenas **14 dos 76 atributos originais**, que são justamente os mais relevantes segundo estudos anteriores (Detrano et al., 1989).

Além disso, com apenas 13 atributos de entrada e 303 instâncias, a dimensionalidade não é um problema:
- Temos mais instâncias do que atributos (303 >> 13)
- A maldição da dimensionalidade afeta principalmente datasets com muitos atributos relativamente ao número de amostras

Portanto, **não apliquei redução de dimensionalidade** (PCA, seleção de atributos, etc.), pois:
1. O dataset já é pré-selecionado pelos especialistas
2. Todos os 13 atributos têm relevância clínica para o diagnóstico
3. A quantidade de atributos é pequena o suficiente para os algoritmos escolhidos

In [ ]:
print('Análise de dimensionalidade:')
print(f'  Número de atributos de entrada: {len(NOME_COLUNAS) - 1}')
print(f'  Número de instâncias: {len(dados_norm)}')
print(f'  Proporção instâncias/atributos: {len(dados_norm) / (len(NOME_COLUNAS) - 1):.1f}')
print()
print('Conclusão: redução de dimensionalidade NÃO é necessária.')
print('Os 13 atributos de entrada são clinicamente relevantes e a base não sofre')
print('da maldição da dimensionalidade (303 instâncias >> 13 atributos).')

Análise de dimensionalidade:
  Número de atributos de entrada: 13
  Número de instâncias: 303
  Proporção instâncias/atributos: 23.3

Conclusão: redução de dimensionalidade NÃO é necessária.
Os 13 atributos de entrada são clinicamente relevantes e a base não sofre
da maldição da dimensionalidade (303 instâncias >> 13 atributos).


---

# Fim da Etapa de Pré-processamento

Os dados estão prontos para a modelagem preditiva. Resumo do que foi feito:

| Etapa | Status |
|-------|--------|
| 1. Identificação do atributo alvo | [OK] `num` -> binário (0 = saudável, 1-4 = doente) |
| 2. Identificação dos tipos de dados | [OK] Quantitativo/Qualitativo |
| 3. Identificação das escalas | [OK] Nominal/Ordinal/Intervalar/Racional |
| 4. Medidas de localidade | [OK] Média, mediana, moda |
| 5. Medidas de espalhamento | [OK] Amplitude, variância, desvio padrão |
| 6. Medidas de distribuição | [OK] Min, Max, Q1, Q2, Q3, IQR |
| 7. Separação do conjunto de teste | [OK] k-fold cross-validation (k=5) |
| 8. Eliminação de atributos desnecessários | [OK] Nenhum foi removido (todos relevantes) |
| 9. Eliminação de exemplos desnecessários | [OK] Sem duplicatas, outliers mantidos |
| 10. Amostragem | [OK] k-fold cross-validation (k=5) |
| 11. Desbalanceamento | [OK] Classes razoavelmente balanceadas |
| 12. Limpeza | [OK] Outliers (IQR), faltantes (mediana/moda) |
| 13. Conversão de tipos | [OK] Min-Max normalization nos quantitativos |
| 14. Redução de dimensionalidade | [OK] Não necessária (13 atributos, 303 instâncias) |

>  **Sobre a não remoção de atributos:** Não removi nenhum atributo porque não entendo de medicina e não sei o que é importante ou não. Por exemplo, `ca` é quantos vasos do coração aparecem num exame de raio-x (pode ser 0, 1, 2 ou 3) e `thal` é um tipo de exame que mostra se o coração está normal, com um defeito fixo ou com um defeito que pode melhorar. Como não sei o significado disso, preferi manter tudo senão poderia jogar fora algo importante para o diagnóstico.

---
# PARTE 2: ANÁLISE PREDITIVA

Nesta segunda parte, vou construir modelos preditivos para classificar se um paciente possui ou não doença cardíaca.

**Técnica de validação escolhida: k-fold cross-validation com k=5**

Por que k-fold e não outras técnicas?
- **Hold-out (70/30)**: depende muito de como os dados são divididos. O resultado pode variar bastante dependendo de quais instâncias vão para teste.
- **Leave-One-Out (LOO)**: treina 303 modelos (um para cada instância). Muito custoso computacionalmente para esta base.
- **k-fold (k=5)**: divide os dados em 5 partes, treina em 4 e testa em 1, repetindo 5 vezes. Cada instância é testada exatamente uma vez. Dá uma estimativa mais estável e confiável do desempenho.

**Modelos testados:**
1. Baseline (classe majoritária) - implementado manualmente
2. KNN (k=5) - scikit-learn
3. Árvore de Decisão (max_depth=5) - scikit-learn
4. MLP (2 camadas ocultas: 10 e 5 neurônios) - scikit-learn

**Métricas de avaliação:** implementadas manualmente (sem sklearn.metrics)

---
## 15. Métricas de Avaliação (Implementação Manual)

Implementei manualmente as métricas de avaliação para analisar o desempenho dos modelos:

- **Matriz de Confusão**: tabela 2×2 mostrando VP, VN, FP, FN
  - VP (Verdadeiro Positivo): doente classificado como doente
  - VN (Verdadeiro Negativo): saudável classificado como saudável
  - FP (Falso Positivo): saudável classificado como doente
  - FN (Falso Negativo): doente classificado como saudável

- **Acurácia**: (VP + VN) / (VP + VN + FP + FN)  — proporção de acertos
- **Precisão**: VP / (VP + FP)  — quantos diagnosticados como doentes realmente estão
- **Recall (Revocação)**: VP / (VP + FN)  — quantos doentes foram identificados
- **F1-Score**: 2 × (precisão × recall) / (precisão + recall)  — média harmônica entre precisão e recall

In [ ]:
def matriz_confusao(y_real, y_pred):
    """Calcula a matriz de confusão 2x2 manualmente."""
    vp = 0  # Verdadeiro Positivo: real=1, pred=1
    vn = 0  # Verdadeiro Negativo: real=0, pred=0
    fp = 0  # Falso Positivo: real=0, pred=1
    fn = 0  # Falso Negativo: real=1, pred=0
    
    for real, pred in zip(y_real, y_pred):
        if real == 1 and pred == 1:
            vp += 1
        elif real == 0 and pred == 0:
            vn += 1
        elif real == 0 and pred == 1:
            fp += 1
        elif real == 1 and pred == 0:
            fn += 1
    
    return vp, vn, fp, fn

def acuracia(vp, vn, fp, fn):
    """Acurácia: proporção de acertos sobre o total."""
    total = vp + vn + fp + fn
    if total == 0:
        return 0.0
    return (vp + vn) / total

def precisao(vp, fp):
    """Precisão: dos que foram classificados como positivos, quantos realmente são."""
    if vp + fp == 0:
        return 0.0
    return vp / (vp + fp)

def recall(vp, fn):
    """Recall (revocação/sensibilidade): dos que são positivos, quantos foram identificados."""
    if vp + fn == 0:
        return 0.0
    return vp / (vp + fn)

def f1_score(prec, rec):
    """F1-Score: média harmônica entre precisão e recall."""
    if prec + rec == 0:
        return 0.0
    return 2 * (prec * rec) / (prec + rec)

def exibir_metricas(y_real, y_pred, nome_modelo):
    """Calcula e exibe todas as métricas para um modelo."""
    vp, vn, fp, fn = matriz_confusao(y_real, y_pred)
    acc = acuracia(vp, vn, fp, fn) * 100
    prec = precisao(vp, fp) * 100
    rec = recall(vp, fn) * 100
    f1 = f1_score(precisao(vp, fp), recall(vp, fn)) * 100
    
    print(f'  {'='*40}')
    print(f'  {nome_modelo}')
    print(f'  {'='*40}')
    print(f'  Matriz de Confusão:')
    print(f'                  Predito')
    print(f'                  Doente  Saudável')
    print(f'  Real  Doente    {vp:4d}     {fn:4d}')
    print(f'        Saudável  {fp:4d}     {vn:4d}')
    print()
    print(f'  Acurácia.: {acc:.2f}%')
    print(f'  Precisão.: {prec:.2f}%')
    print(f'  Recall...: {rec:.2f}%')
    print(f'  F1-Score.: {f1:.2f}%')
    print()
    
    return {'acc': acc, 'prec': prec, 'rec': rec, 'f1': f1, 'vp': vp, 'vn': vn, 'fp': fp, 'fn': fn}

---
## 16. K-Fold Cross-Validation (Implementação Manual)

Implementei o k-fold cross-validation manualmente. A lógica é:
1. Embaralhar os dados com uma seed fixa (para reprodutibilidade)
2. Dividir em k partes (folds) de tamanho aproximadamente igual
3. Para cada fold:
   - Usar o fold como teste
   - Usar os k-1 folds restantes como treino
   - Treinar o modelo e avaliar
4. Calcular a média e o desvio padrão das métricas entre os k folds

Escolhi k=5 porque:
- É o valor mais comum na literatura (equilíbrio entre viés e variância)
- Cada fold de teste tem ~60 instâncias, suficiente para métricas estáveis
- São apenas 5 treinamentos, diferente do LOO que seriam 303

In [ ]:
def kfold_split(dados, k=5, seed=42):
    """Divide os dados em k folds de forma estratificada.
    Estratificada = mantém a proporção de classes em cada fold."""
    # Separar por classe
    classe_0 = [i for i, inst in enumerate(dados) if inst[INDICE_ALVO] == 0]
    classe_1 = [i for i, inst in enumerate(dados) if inst[INDICE_ALVO] != 0]
    
    # Embaralhar com seed fixa
    random.seed(seed)
    random.shuffle(classe_0)
    random.shuffle(classe_1)
    
    # Dividir cada classe em k partes
    folds_0 = []
    for i in range(k):
        inicio = (i * len(classe_0)) // k
        fim = ((i + 1) * len(classe_0)) // k
        folds_0.append(classe_0[inicio:fim])
    
    folds_1 = []
    for i in range(k):
        inicio = (i * len(classe_1)) // k
        fim = ((i + 1) * len(classe_1)) // k
        folds_1.append(classe_1[inicio:fim])
    
    # Juntar as classes para formar cada fold
    folds = []
    for i in range(k):
        fold = folds_0[i] + folds_1[i]
        random.shuffle(fold)
        folds.append(fold)
    
    return folds

def preparar_dados_kfold(dados_norm, folds, fold_idx):
    """Prepara os dados para um fold específico.
    fold_idx: índice do fold que será usado como teste."""
    indices_treino = []
    indices_teste = []
    
    for i, fold in enumerate(folds):
        if i == fold_idx:
            indices_teste.extend(fold)
        else:
            indices_treino.extend(fold)
    
    # Separar atributos (X) e alvo (y)
    X_treino = []
    y_treino = []
    for idx in indices_treino:
        X_treino.append(dados_norm[idx][:INDICE_ALVO])
        # Binarizar alvo: 0 = saudável, >0 = doente
        y_treino.append(0 if dados_norm[idx][INDICE_ALVO] == 0 else 1)
    
    X_teste = []
    y_teste = []
    for idx in indices_teste:
        X_teste.append(dados_norm[idx][:INDICE_ALVO])
        y_teste.append(0 if dados_norm[idx][INDICE_ALVO] == 0 else 1)
    
    return X_treino, y_treino, X_teste, y_teste

# Testando a divisão
folds = kfold_split(dados_norm, k=5)
print(f'Total de folds: {len(folds)}')
print()
for i, fold in enumerate(folds):
    print(f'  Fold {i+1}: {len(fold)} instâncias')
print()
print(f'Total: {sum(len(f) for f in folds)} instâncias')

Total de folds: 5

  Fold 1: 59 instâncias
  Fold 2: 61 instâncias
  Fold 3: 61 instâncias
  Fold 4: 61 instâncias
  Fold 5: 61 instâncias

Total: 303 instâncias


---
## 15b. Configuração dos Experimentos

Na célula abaixo ficam as configurações dos experimentos: valores de k do KNN e uso do Leave-One-Out.

In [ ]:
# ===== CONFIGURACAO DOS EXPERIMENTOS =====
# Altere os valores abaixo para testar outras configuracoes

K_VALUES = [5]                  # valores de k para o KNN (ex: [3, 5, 7])
USAR_LOO = True                 # Leave-One-Out (mais lento: 303 treinamentos)
COMPARAR_TREINO_TESTE = True    # compara acuracia de treino vs teste (overfitting)

print('Configuracao dos experimentos:')
print(f'  KNN: k = {K_VALUES}')
print(f'  Leave-One-Out: {"Sim" if USAR_LOO else "Nao"}')
print(f'  Comparar treino vs teste: {"Sim" if COMPARAR_TREINO_TESTE else "Nao"}')


Configuracao dos experimentos:
  KNN: k = [5]
  Leave-One-Out: Sim
  Comparar treino vs teste: Sim


---
## 15c. Leave-One-Out Cross-Validation (Implementação Manual)

O **Leave-One-Out (LOO)** é um caso extremo do k-fold: cada fold tem apenas 1 instância de teste.
Treinamos 303 modelos — cada um com 302 amostras de treino e 1 de teste.

**Vantagens:**
- Praticamente sem viés (usa quase todos os dados)
- Determinístico

**Desvantagens:**
- Muito custoso (303 treinamentos)
- Pode ter alta variância

In [ ]:
def leave_one_out_split(dados):
    n = len(dados)
    folds = []
    for i in range(n):
        folds.append([i])
    return folds

def preparar_dados_loo(dados_norm, indice_teste):
    X_treino = []
    y_treino = []
    X_teste = []
    y_teste = []
    for i, inst in enumerate(dados_norm):
        if i == indice_teste:
            X_teste.append(inst[:INDICE_ALVO])
            y_teste.append(0 if inst[INDICE_ALVO] == 0 else 1)
        else:
            X_treino.append(inst[:INDICE_ALVO])
            y_treino.append(0 if inst[INDICE_ALVO] == 0 else 1)
    return X_treino, y_treino, X_teste, y_teste

def acuracia_treino(modelo, X_treino, y_treino):
    y_pred = modelo.predict(X_treino)
    vp, vn, fp, fn = matriz_confusao(y_treino, y_pred)
    return acuracia(vp, vn, fp, fn) * 100

print('[OK] Funcoes para LOO e acuracia de treino carregadas.')

[OK] Funcoes para LOO e acuracia de treino carregadas.


In [ ]:
def executar_knn(dados_norm, folds, k, usar_loo=False):
    """Executa KNN com k-fold ou LOO e retorna acuracias."""
    acuracias_teste = []
    acuracias_treino = []

    if usar_loo:
        loo_folds = leave_one_out_split(dados_norm)
        for i in range(len(loo_folds)):
            X_treino, y_treino, X_teste, y_teste = preparar_dados_loo(dados_norm, i)
            modelo = KNeighborsClassifier(n_neighbors=k)
            modelo.fit(X_treino, y_treino)
            y_pred = modelo.predict(X_teste)
            vp, vn, fp, fn = matriz_confusao(y_teste, y_pred)
            acuracias_teste.append(acuracia(vp, vn, fp, fn) * 100)
            if COMPARAR_TREINO_TESTE:
                acuracias_treino.append(acuracia_treino(modelo, X_treino, y_treino))
    else:
        for i in range(len(folds)):
            X_treino, y_treino, X_teste, y_teste = preparar_dados_kfold(dados_norm, folds, i)
            modelo = KNeighborsClassifier(n_neighbors=k)
            modelo.fit(X_treino, y_treino)
            y_pred = modelo.predict(X_teste)
            vp, vn, fp, fn = matriz_confusao(y_teste, y_pred)
            acuracias_teste.append(acuracia(vp, vn, fp, fn) * 100)
            if COMPARAR_TREINO_TESTE:
                acuracias_treino.append(acuracia_treino(modelo, X_treino, y_treino))

    return acuracias_treino, acuracias_teste

print('Funcao executar_knn carregada.')

Funcao executar_knn carregada.


---
## 15d. EXECUTAR EXPERIMENTOS

Execute esta celula para rodar APENAS os experimentos que voce escolheu no menu acima.

In [ ]:
print('=' * 70)
print('EXECUTANDO EXPERIMENTOS SELECIONADOS')
print('=' * 70)
print()

resultados_loo = {}
resultados_knn = {}

# === KNN K-FOLD ===
if K_VALUES:
    print('--- KNN com k-fold ---')
    print()
    for k in K_VALUES:
        print(f'  Treinando KNN com k={k}...')
        accs_treino, accs_teste = executar_knn(dados_norm, folds, k, usar_loo=False)
        med_teste = media(accs_teste)
        dp_teste = desvio_padrao(accs_teste)
        med_treino = media(accs_treino) if accs_treino else 0
        resultados_knn[k] = (med_treino, med_teste, dp_teste)
        print(f'    Acuracia teste: {med_teste:.2f}% +- {dp_teste:.2f}%')
        if COMPARAR_TREINO_TESTE and accs_treino:
            dif = med_treino - med_teste
            print(f'    Acuracia treino: {med_treino:.2f}%')
            print(f'    Diferenca: {dif:+.2f} p.p. ', end='')
            if dif > 10:
                print('[OVERFITTING!]')
            elif dif > 5:
                print('[Leve overfitting]')
            else:
                print('[OK - generaliza bem]')
        print()
    if len(K_VALUES) > 1:
        print('  Comparacao:')
        melhor_k = max(K_VALUES, key=lambda k: resultados_knn[k][1])
        for k in K_VALUES:
            _, med_teste, _ = resultados_knn[k]
            print(f'    k={k}: {med_teste:.2f}%')
        print(f'  -> Melhor k: {melhor_k} ({resultados_knn[melhor_k][1]:.2f}%)')
        print()

# === LEAVE-ONE-OUT ===
k_values_loo = K_VALUES if K_VALUES else [5]
if USAR_LOO:
    print('--- Leave-One-Out ---')
    print('  (Processando 303 iteracoes... pode levar alguns minutos)')
    print()
    for k in k_values_loo:
        print(f'  KNN k={k} com LOO...')
        accs_treino, accs_teste = executar_knn(dados_norm, None, k, usar_loo=True)
        med_teste = media(accs_teste)
        dp_teste = desvio_padrao(accs_teste)
        resultados_loo[k] = (media(accs_treino) if accs_treino else 0, med_teste, dp_teste)
        print(f'    Acuracia LOO (teste): {med_teste:.2f}% +- {dp_teste:.2f}%')
        if COMPARAR_TREINO_TESTE and accs_treino:
            med_treino_loo = media(accs_treino)
            print(f'    Acuracia LOO (treino): {med_treino_loo:.2f}%')
            print(f'    Diferenca (treino - teste): {med_treino_loo - med_teste:+.2f} p.p.')
        if K_VALUES:
            _, acc_kfold, _ = resultados_knn.get(k, (0, 0, 0))
            if acc_kfold > 0:
                print(f'    Acuracia k-fold: {acc_kfold:.2f}%')
                print(f'    Diferenca LOO vs k-fold: {med_teste - acc_kfold:+.2f} p.p.')
        print()
else:
    print('[Leave-One-Out nao selecionado]')
    print()

# === KNN DETALHADO (k-fold, confusion matrix por fold) ===
todas_metricas_knn = []
k_detalhado = K_VALUES[0] if K_VALUES else 5
if K_VALUES:
    for i in range(5):
        X_treino, y_treino, X_teste, y_teste = preparar_dados_kfold(dados_norm, folds, i)
        modelo = KNeighborsClassifier(n_neighbors=k_detalhado)
        modelo.fit(X_treino, y_treino)
        y_pred = modelo.predict(X_teste)
        metricas = exibir_metricas(y_teste, y_pred, f'Fold {i+1} - KNN (k={k_detalhado})')
        todas_metricas_knn.append(metricas)
        print()
    media_acc = media([m['acc'] for m in todas_metricas_knn])
    media_prec = media([m['prec'] for m in todas_metricas_knn])
    media_rec = media([m['rec'] for m in todas_metricas_knn])
    media_f1 = media([m['f1'] for m in todas_metricas_knn])
    print(f'  {"="*40}')
    print(f'  MEDIA KNN (k={k_detalhado}, 5 folds)')
    print(f'  {"="*40}')
    print(f'  Acuracia media.: {media_acc:.2f}%')
    print(f'  Precisao media.: {media_prec:.2f}%')
    print(f'  Recall medio...: {media_rec:.2f}%')
    print(f'  F1 medio.......: {media_f1:.2f}%')
    print()

# === ARVORE DE DECISAO ===
todas_metricas_dt = []
for i in range(5):
    X_treino, y_treino, X_teste, y_teste = preparar_dados_kfold(dados_norm, folds, i)
    modelo = DecisionTreeClassifier(max_depth=5, random_state=42)
    modelo.fit(X_treino, y_treino)
    y_pred = modelo.predict(X_teste)
    metricas = exibir_metricas(y_teste, y_pred, f'Fold {i+1} - Arvore Decisao')
    todas_metricas_dt.append(metricas)
    print()
media_acc = media([m['acc'] for m in todas_metricas_dt])
media_prec = media([m['prec'] for m in todas_metricas_dt])
media_rec = media([m['rec'] for m in todas_metricas_dt])
media_f1 = media([m['f1'] for m in todas_metricas_dt])
print(f'  {"="*40}')
print(f'  MEDIA ARVORE DE DECISAO (5 folds)')
print(f'  {"="*40}')
print(f'  Acuracia media.: {media_acc:.2f}%')
print(f'  Precisao media.: {media_prec:.2f}%')
print(f'  Recall medio...: {media_rec:.2f}%')
print(f'  F1 medio.......: {media_f1:.2f}%')
print()

# === MLP ===
todas_metricas_mlp = []
for i in range(5):
    X_treino, y_treino, X_teste, y_teste = preparar_dados_kfold(dados_norm, folds, i)
    modelo = MLPClassifier(hidden_layer_sizes=(10, 5), max_iter=1000, random_state=42)
    modelo.fit(X_treino, y_treino)
    y_pred = modelo.predict(X_teste)
    metricas = exibir_metricas(y_teste, y_pred, f'Fold {i+1} - MLP')
    todas_metricas_mlp.append(metricas)
    print()
media_acc = media([m['acc'] for m in todas_metricas_mlp])
media_prec = media([m['prec'] for m in todas_metricas_mlp])
media_rec = media([m['rec'] for m in todas_metricas_mlp])
media_f1 = media([m['f1'] for m in todas_metricas_mlp])
print(f'  {"="*40}')
print(f'  MEDIA MLP (5 folds)')
print(f'  {"="*40}')
print(f'  Acuracia media.: {media_acc:.2f}%')
print(f'  Precisao media.: {media_prec:.2f}%')
print(f'  Recall medio...: {media_rec:.2f}%')
print(f'  F1 medio.......: {media_f1:.2f}%')
print()

# === TABELA COMPARATIVA FINAL ===
print()
print('=' * 80)
print('TABELA COMPARATIVA FINAL')
print('=' * 80)
print()
print(f'{"Experimento":30s} {"Acuracia":20s} {"Treino":20s}')
print('-' * 80)
for k in K_VALUES:
    if k in resultados_knn:
        acc_treino, acc_teste, dp_teste = resultados_knn[k]
        treino_str = f'{acc_treino:.2f}%' if COMPARAR_TREINO_TESTE else '-'
        print(f'{"KNN k-fold (k=" + str(k) + ")":30s} {acc_teste:.2f}% +-{dp_teste:.2f}           {treino_str:>20s}')
if USAR_LOO:
    for k in k_values_loo:
        if k in resultados_loo:
            acc_treino_loo, med_loo, dp_loo = resultados_loo[k]
            _, acc_kfold, _ = resultados_knn.get(k, (0, 0, 0))
            treino_str = f'{acc_treino_loo:.2f}%' if COMPARAR_TREINO_TESTE else '-'
            print(f'{"KNN LOO (k=" + str(k) + ")":30s} {med_loo:.2f}% +-{dp_loo:.2f}           {treino_str:>20s}  (k-fold: {acc_kfold:.2f}%)')
if todas_metricas_dt:
    med_dt = media([m['acc'] for m in todas_metricas_dt])
    dp_dt = desvio_padrao([m['acc'] for m in todas_metricas_dt])
    print(f'{"Arvore Decisao (max_depth=5)":30s} {med_dt:.2f}% +-{dp_dt:.2f}           (veja detalhes acima)')
if todas_metricas_mlp:
    med_mlp = media([m['acc'] for m in todas_metricas_mlp])
    dp_mlp = desvio_padrao([m['acc'] for m in todas_metricas_mlp])
    print(f'{"MLP (10,5)":30s} {med_mlp:.2f}% +-{dp_mlp:.2f}           (veja detalhes acima)')
print()
print('Interpretacao:')
print('  - Diferenca treino vs teste pequena = modelo generaliza bem')
print('  - Diferenca treino >> teste = overfitting')
if USAR_LOO:
    print('  - LOO vs k-fold parecidos = validacao consistente')


EXECUTANDO EXPERIMENTOS SELECIONADOS

--- KNN com k-fold ---

  Treinando KNN com k=5...
    Acuracia teste: 82.86% +- 5.18%
    Acuracia treino: 84.82%
    Diferenca: +1.96 p.p. [OK - generaliza bem]

--- Leave-One-Out ---
  (Processando 303 iteracoes... pode levar alguns minutos)

  KNN k=5 com LOO...
    Acuracia LOO (teste): 82.18% +- 38.33%
    Acuracia LOO (treino): 84.18%
    Diferenca (treino - teste): +2.00 p.p.
    Acuracia k-fold: 82.86%
    Diferenca LOO vs k-fold: -0.68 p.p.

  Fold 1 - KNN (k=5)
  Matriz de Confusão:
                  Predito
                  Doente  Saudável
  Real  Doente      24        3
        Saudável     5       27

  Acurácia.: 86.44%
  Precisão.: 82.76%
  Recall...: 88.89%
  F1-Score.: 85.71%


  Fold 2 - KNN (k=5)
  Matriz de Confusão:
                  Predito
                  Doente  Saudável
  Real  Doente      24        4
        Saudável     2       31

  Acurácia.: 90.16%
  Precisão.: 92.31%
  Recall...: 85.71%
  F1-Score.: 88.89%


  Fo

  Fold 2 - MLP
  Matriz de Confusão:
                  Predito
                  Doente  Saudável
  Real  Doente      23        5
        Saudável     3       30

  Acurácia.: 86.89%
  Precisão.: 88.46%
  Recall...: 82.14%
  F1-Score.: 85.19%




  Fold 3 - MLP
  Matriz de Confusão:
                  Predito
                  Doente  Saudável
  Real  Doente      20        8
        Saudável     8       25

  Acurácia.: 73.77%
  Precisão.: 71.43%
  Recall...: 71.43%
  F1-Score.: 71.43%


  Fold 4 - MLP
  Matriz de Confusão:
                  Predito
                  Doente  Saudável
  Real  Doente      23        5
        Saudável     6       27

  Acurácia.: 81.97%
  Precisão.: 79.31%
  Recall...: 82.14%
  F1-Score.: 80.70%


  Fold 5 - MLP
  Matriz de Confusão:
                  Predito
                  Doente  Saudável
  Real  Doente      24        4
        Saudável     7       26

  Acurácia.: 81.97%
  Precisão.: 77.42%
  Recall...: 85.71%
  F1-Score.: 81.36%


  MEDIA MLP (5 folds)
  Acuracia media.: 81.19%
  Precisao media.: 79.48%
  Recall medio...: 79.84%
  F1 medio.......: 79.58%


TABELA COMPARATIVA FINAL

Experimento                    Acuracia             Treino              
--------------------------------------

---
## 17. Classificador Baseline (Classe Majoritária)

O baseline é o modelo mais simples possível: ele sempre prevê a classe mais frequente no conjunto de treino. 

**Por que usar um baseline?**
Qualquer modelo que eu criar precisa ter desempenho melhor que o baseline para ser considerado útil. Se o baseline já tem 60% de acurácia (por exemplo), o KNN precisa entregar mais que isso.

O baseline escolhido é o **algoritmo da classe majoritária**: contar quantos pacientes estão em cada classe no treino e prever sempre a classe com mais instâncias.

In [ ]:
def baseline_majoritario(y_treino, y_teste):
    """Baseline: prevê sempre a classe majoritária do treino."""
    contagem = Counter(y_treino)
    classe_majoritaria = contagem.most_common(1)[0][0]
    return [classe_majoritaria] * len(y_teste)

print('=' * 60)
print('CLASSIFICADOR BASELINE (CLASSE MAJORITÁRIA)')
print('=' * 60)
print()

resultados_baseline = []

for i in range(5):
    X_treino, y_treino, X_teste, y_teste = preparar_dados_kfold(dados_norm, folds, i)
    
    contagem_treino = Counter(y_treino)
    classe_majoritaria = contagem_treino.most_common(1)[0][0]
    
    y_pred = baseline_majoritario(y_treino, y_teste)
    
    vp, vn, fp, fn = matriz_confusao(y_teste, y_pred)
    acc = acuracia(vp, vn, fp, fn) * 100
    resultados_baseline.append(acc)
    
    print(f'Fold {i+1}: majoritária = classe {classe_majoritaria} | acurácia = {acc:.2f}%')

print()
media_baseline = media(resultados_baseline)
dp_baseline = desvio_padrao(resultados_baseline)
print(f'Média da acurácia baseline (5 folds): {media_baseline:.2f}% ± {dp_baseline:.2f}%')
print()
print('O baseline serve como referência mínima. Modelos melhores precisam superar este valor.')

CLASSIFICADOR BASELINE (CLASSE MAJORITÁRIA)

Fold 1: majoritária = classe 0 | acurácia = 54.24%
Fold 2: majoritária = classe 0 | acurácia = 54.10%
Fold 3: majoritária = classe 0 | acurácia = 54.10%
Fold 4: majoritária = classe 0 | acurácia = 54.10%
Fold 5: majoritária = classe 0 | acurácia = 54.10%

Média da acurácia baseline (5 folds): 54.13% ± 0.06%

O baseline serve como referência mínima. Modelos melhores precisam superar este valor.


---
## 21. Análise dos Resultados

Agora vou comparar o desempenho de todos os modelos lado a lado para tirar conclusões.

In [ ]:
# Tabela comparativa consolidada
# Os resultados completos estao na celula EXECUTAR EXPERIMENTOS acima.
# Esta celula apenas exibe o resumo final.
print()
print('=' * 80)
print('RESUMO FINAL (detalhes no experimento acima)')
print('=' * 80)
print()

# Baseline
accs_baseline = []
precs_baseline = []
recs_baseline = []
f1s_baseline = []
for i in range(5):
    X_treino, y_treino, X_teste, y_teste = preparar_dados_kfold(dados_norm, folds, i)
    y_pred = baseline_majoritario(y_treino, y_teste)
    vp, vn, fp, fn = matriz_confusao(y_teste, y_pred)
    acc = acuracia(vp, vn, fp, fn) * 100
    prec = precisao(vp, fp) * 100
    rec = recall(vp, fn) * 100
    f1 = f1_score(precisao(vp, fp), recall(vp, fn)) * 100
    accs_baseline.append(acc)
    precs_baseline.append(prec)
    recs_baseline.append(rec)
    f1s_baseline.append(f1)

print(f'{"Modelo":20s} {"Acuracia":18s} {"Precisao":18s} {"Recall":18s} {"F1-Score":18s}')
print('-' * 80)

modelos = [
    ('Baseline', accs_baseline, precs_baseline, recs_baseline, f1s_baseline),
]

if todas_metricas_knn:
    modelos.append(('KNN (k=' + str(K_VALUES[0]) + ')',
        [m['acc'] for m in todas_metricas_knn],
        [m['prec'] for m in todas_metricas_knn],
        [m['rec'] for m in todas_metricas_knn],
        [m['f1'] for m in todas_metricas_knn]))
for k in K_VALUES:
    if k != K_VALUES[0] and k in resultados_knn:
        _, med_teste, _ = resultados_knn[k]
        modelos.append((f'KNN (k={k})', [med_teste], [0], [0], [0]))

if todas_metricas_dt:
    modelos.append(('Arvore Decisao',
        [m['acc'] for m in todas_metricas_dt],
        [m['prec'] for m in todas_metricas_dt],
        [m['rec'] for m in todas_metricas_dt],
        [m['f1'] for m in todas_metricas_dt]))
if todas_metricas_mlp:
    modelos.append(('MLP (10,5)',
        [m['acc'] for m in todas_metricas_mlp],
        [m['prec'] for m in todas_metricas_mlp],
        [m['rec'] for m in todas_metricas_mlp],
        [m['f1'] for m in todas_metricas_mlp]))

for nome, accs, precs, recs, f1s in modelos:
    m_acc = media(accs)
    dp_acc = desvio_padrao(accs) if len(accs) > 1 else 0
    m_prec = media(precs) if len(precs) > 1 else 0
    m_rec = media(recs) if len(recs) > 1 else 0
    m_f1 = media(f1s) if len(f1s) > 1 else 0
    print(f'{nome:20s} {m_acc:6.2f}% +- {dp_acc:4.2f}%  {m_prec:6.2f}%       {m_rec:6.2f}%       {m_f1:6.2f}%')

print()
print('=' * 80)
print()
print('Nota: Para ver matrizes de confusao detalhadas de cada fold,')
print('veja a celula EXECUTAR EXPERIMENTOS (secao 15d).')



RESUMO FINAL (detalhes no experimento acima)

Modelo               Acuracia           Precisao           Recall             F1-Score          
--------------------------------------------------------------------------------
Baseline              54.13% +- 0.06%    0.00%         0.00%         0.00%
KNN (k=5)             82.86% +- 5.18%   82.35%        79.92%        80.97%
Arvore Decisao        77.22% +- 5.13%   76.16%        73.39%        74.63%
MLP (10,5)            81.19% +- 4.71%   79.48%        79.84%        79.58%


Nota: Para ver matrizes de confusao detalhadas de cada fold,
veja a celula EXECUTAR EXPERIMENTOS (secao 15d).


In [ ]:
# ===== ANALISE DE IMPORTANCIA DOS ATRIBUTOS =====
# Vou usar a Arvore de Decisao (ja treinada) para ver
# quais atributos sao mais importantes para o diagnostico

print('=' * 70)
print('ANALISE DE IMPORTANCIA DOS ATRIBUTOS')
print('=' * 70)
print()
print('Usei uma Arvore de Decisao treinada com TODOS os dados para')
print('medir o quanto cada atributo contribui para a decisao final:')
print()

X_total = [inst[:INDICE_ALVO] for inst in dados_norm]
y_total = [0 if inst[INDICE_ALVO] == 0 else 1 for inst in dados_norm]

arvore = DecisionTreeClassifier(max_depth=5, random_state=42)
arvore.fit(X_total, y_total)

importancias = arvore.feature_importances_

pares = list(zip(NOME_COLUNAS[:INDICE_ALVO], importancias))
pares.sort(key=lambda x: x[1], reverse=True)

print(f'  {"Atributo":12s}  Importancia')
print(f'  {"-"*25}')
for nome, imp in pares:
    barra = '█' * int(imp * 40) + '░' * (40 - int(imp * 40))
    print(f'  {nome:12s}  {imp*100:5.1f}%  {barra}')

print()
print('  Interpretacao:')
print('  - Quanto maior a %, mais o atributo ajuda na decisao')
print('  - Atributos com 0.0% nao foram usados pela arvore')

print()
print('Correlacao de cada atributo com o alvo (doenca cardiaca):')
print('  (positivo = quanto maior o valor, mais chance de ter doenca)')
print()

correlacoes = []
for j in range(INDICE_ALVO):
    x_vals = [inst[j] for inst in dados_norm]
    n = len(x_vals)
    media_x = sum(x_vals) / n
    media_y = sum(y_total) / n
    num = sum((x_vals[i] - media_x) * (y_total[i] - media_y) for i in range(n))
    den = math.sqrt(sum((x_vals[i] - media_x) ** 2 for i in range(n))) * \
          math.sqrt(sum((y_total[i] - media_y) ** 2 for i in range(n)))
    corr = num / den if den != 0 else 0
    correlacoes.append((NOME_COLUNAS[j], corr))

correlacoes.sort(key=lambda x: abs(x[1]), reverse=True)

print(f'  {"Atributo":12s}  Correlacao')
print(f'  {"-"*25}')
for nome, corr in correlacoes:
    print(f'  {nome:12s}  {corr:+.4f}')
print()

print('Exemplos de leitura:')
print('- "thal" com correlacao positiva alta: quanto maior o valor, mais associado a doenca')
print('- "ca" com correlacao positiva: mais vasos afetados = maior chance de doenca')
print('Isto ajuda a entender quais exames sao mais importantes para o diagnostico.')

ANALISE DE IMPORTANCIA DOS ATRIBUTOS

Usei uma Arvore de Decisao treinada com TODOS os dados para
medir o quanto cada atributo contribui para a decisao final:

  Atributo      Importancia
  -------------------------
  thal           36.6%  ██████████████░░░░░░░░░░░░░░░░░░░░░░░░░░
  cp             17.8%  ███████░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░
  ca             13.1%  █████░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░
  thalach         6.1%  ██░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░
  chol            5.8%  ██░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░
  age             5.4%  ██░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░
  oldpeak         4.9%  █░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░
  trestbps        3.4%  █░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░
  slope           3.0%  █░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░
  restecg         2.1%  ░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░
  sex             1.9%  ░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░
  fbs             0.0%  ░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░
  ex

---

# Conclusão

Neste trabalho, realizei o pré-processamento completo e a análise preditiva sobre a base de dados Heart Disease (Cleveland).

**Principais etapas do pré-processamento:**
- Carregamento e exploração manual dos dados
- Identificação de tipos, escalas e medidas estatísticas
- Tratamento de valores ausentes (mediana para `ca`, moda para `thal`)
- Detecção de outliers pelo método IQR (mantidos por serem dados clínicos reais)
- Normalização Min-Max para preparar os dados para o KNN
- Binarização do alvo (0 = saudável, 1-4 = doente)

**Resultados obtidos (k-fold, k=5):**

| Modelo | Acurácia | Precisão | Recall | F1-Score |
|--------|----------|----------|--------|----------|
| Baseline | 54.13% | 0.00% | 0.00% | 0.00% |
| KNN (k=5) | **82.86%** | **82.35%** | 79.92% | **80.97%** |
| Árvore de Decisão | 77.22% | 76.16% | 73.39% | 74.63% |
| MLP (10, 5) | 81.19% | 79.48% | **79.84%** | 79.58% |

O **KNN com k=5 e k-fold cross-validation** foi o modelo com melhor desempenho geral, atingindo 82.86% de acurácia e 80.97% de F1-Score, superando o baseline em 28.7 pontos percentuais. O MLP apresentou desempenho próximo (81.19%), enquanto a Árvore de Decisão ficou ligeiramente abaixo (77.22%).

Além do k-fold, foi testado o Leave-One-Out como método alternativo de validação para o KNN, que apresentou resultados consistentes com o k-fold, confirmando a estabilidade da avaliação.

Todas as métricas (acurácia, precisão, recall, F1-score) foram implementadas manualmente, sem uso de bibliotecas externas.